In [3]:
def create_outline(state):
    # fetch title
    title = state["title"]

    # call LLM to generate outline
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).content

    # update state
    state["outline"] = outline

    return state

In [13]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    rating: float
    feedback: str   

In [4]:
def create_blog(state):
    # fetch title and outline from the state
    title = state["title"]
    outline = state["outline"]

    # create the prompt
    prompt = (
        f"Write a detailed blog on the title - {title} "
        f"using the following outline:\n\n{outline}"
    )

    # generate the blog content
    content = model.invoke(prompt).content

    # update the state
    state["content"] = content

    return state

In [17]:
def evaluate(state):
    title = state["title"]
    content = state["content"]

    prompt = f"""
You are an expert blog evaluator.

Evaluate the following blog.

Title:
{title}

Blog:
{content}

Return ONLY valid JSON.

{{
    "rating": 8.5,
    "feedback": "Your feedback"
}}
"""

    response = model.invoke(prompt).content

    print(response)

In [18]:
graph= StateGraph(BlogState)

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate',evaluate)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','evaluate')
graph.add_edge('evaluate',END)

workflow = graph.compile()

In [19]:
initial_state = {
    "title": "Sword Art Online X Pancake Collaboration"
}

final_state = workflow.invoke(initial_state)

print(final_state)

```json
{
    "rating": 9.5,
    "feedback": "This blog post is an outstanding example of how to announce a themed collaboration. It's incredibly detailed, imaginative, and perfectly tailored to its target audience. Here's a breakdown:\n\n**Strengths:**\n\n1.  **Exceptional Thematic Integration:** The blog seamlessly weaves the Sword Art Online universe into every aspect of the collaboration. From 'Kirito's Dual Wield Stack' to 'Yui's Pixie Dust Pancakes,' 'Elixir Potions,' 'Aincrad Floor Maps,' and 'Exclusive Loot Drops,' every element feels authentic and well-thought-out. The descriptions of the food, drinks, and decor are vivid and directly reference SAO lore, making it highly appealing to fans.\n2.  **Incredible Detail and Creativity:** The level of detail provided for each menu item (description, toppings, garnish) is fantastic. It goes beyond simple branding, creating a truly immersive experience. The ideas for ambiance (themed lighting, SAO soundtrack, quest board menus), intera

In [20]:
print(final_state['outline'])

Here's a detailed outline for a blog post about a hypothetical (or real, if one exists!) Sword Art Online X Pancake Collaboration. The outline aims to generate excitement, inform, and engage fans of both SAO and delicious breakfast food.

---

## Blog Post Title Ideas:

*   **Level Up Your Breakfast: Announcing the Epic Sword Art Online x [Pancake Chain Name] Collaboration!**
*   **Dive into Aincrad, One Stack at a Time: SAO x Pancakes is Here!**
*   **From Virtual Worlds to Your Plate: The Ultimate SAO Pancake Quest Begins!**
*   **Sword Art Online's Sweetest Quest Yet: A Pancake Collaboration Review!** (If reviewing an existing one)

---

## Detailed Blog Post Outline:

**I. Introduction: Logging In to a Delicious New Adventure**

*   **A. Catchy Hook:**
    *   Start with a question: "Ever wished you could taste the food from Aincrad?" or "What if your favorite virtual world collided with your favorite breakfast?"
    *   Briefly mention the immense popularity and immersive nature o

In [23]:
print(final_state['content'])

## Level Up Your Breakfast: Announcing the Epic Sword Art Online x Golden Griddle Pancakes Collaboration!

Ever wished you could taste the food from Aincrad? What if your favorite virtual world collided with your favorite breakfast? For fans of Sword Art Online, the immersive world of Aincrad, Alfheim, and Underworld has captivated millions with its thrilling adventures, deep character bonds, and yes, even its surprisingly detailed culinary moments. From Asuna's legendary cooking skill to the importance of sustenance in a life-or-death virtual reality, food has always played a subtle yet significant role in the SAO universe.

Get ready, SAO fans and foodies! We're thrilled to announce the groundbreaking **Sword Art Online x Golden Griddle Pancakes Collaboration!** Prepare to log in for a limited-time event that brings the magic of SAO directly to your plate through an incredible array of themed pancakes, elixir drinks, and an immersive dining experience that will transport you straight